# Search for occurrences and omissions of names

## 1 Import needed packages

In [ ]:
import concurrent.futures
import pandas as pd
from tqdm.notebook import tqdm
from utils import process_bkv
from constants import DATA_PATH, TMP_PATH

## 2 Merge verses from na28, ecm, and tei into one verses.csv and add unique identifieres

In [ ]:
verses_na28ecm_df = pd.read_csv(
    TMP_PATH + "na28ecm_verses.csv",
    low_memory=False,
)

verses_df = pd.read_csv(
    TMP_PATH + "verses.csv",
    low_memory=False,
)

verses_df = pd.concat([verses_df, verses_na28ecm_df])
verses_df.drop_duplicates()

# add unique integer verse_id, as the transcription (or metadata like encoding_version or edition_version) can change over time
verses_df["verse_id"] = range(1, len(verses_df) + 1)

verses_df.to_csv(TMP_PATH + "verses.csv", mode="w", encoding="utf8", index=False)


# Set source to 'ecm' where ga == 'ecm'
verses_df.loc[verses_df["ga"] == "ecm", "source"] = "na28"

# Set source to 'na28' where ga == 'na28'
verses_df.loc[verses_df["ga"] == "na28", "source"] = "na28"

## 2 Read data from files

In [ ]:
# data files
verses_data = TMP_PATH + "verses.csv"
word_data = TMP_PATH + "words.csv"

# Read CSV files into a DataFrames
verses_df = pd.read_csv(
    verses_data,
    low_memory=False,
    dtype={
        "ga": "string",
        "bkv": "string",
        "text": "string",
        "transcript": "string",
        "lection": "string",
        "source": "string",
        "verse_id": "Int64",
    },
    usecols=[
        "ga",
        "bkv",
        "text",
        "transcript",
        "lection",
        "source",
        "verse_id",
    ],
)
words_df = pd.read_csv(
    word_data,
    low_memory=False,
    dtype={
        "label:en": "string",
        "label:el:norm": "string",
        "gender": "string",
        "variant": "string",
        "wordID": "Int64",
        "variantID": "Int64",
    },
    usecols=[
        "label:en",
        "label:el:norm",
        "gender",
        "variant",
        "wordID",
        "variantID",
    ],
)

In [ ]:
# import json
# # Load the JSON data from the file
# with open("../data/blacklist.json") as f:
#     blacklist_dict = json.load(f)

# # example json file content for Matt.1.15 and its names (easy way):
# {
#  "blacklist": [
#    {
#     "verseID": 380,
#     "wordIDs": [204,77,123,74]
#    }
#  ]
# }
# # example json file content for Matt.1.15 and its names (how it is robust after rerunning):
# {
#  "blacklist": [
#    {
#       "ga": 2737
#       "lection": ""
#       "nkv": "Matt.1.15"
#       "source": "ntvmr"
#       "label:el:norm": ["","",""]
#    }
#  ]
# }

# # Convert the list of dictionaries into a dictionary with verseID as the key
# # blacklist_dict = {
# #     item["verseID"]: item["wordIDs"] for item in blacklist_dict["blacklist"]
# # }

blacklist_dict = {}

## 3 Search for omissions and occurrences by bkv 

In [ ]:
# overwrite = True

# # Get unique values from the 'bkv' column
# unique_bkvs = ["B01K1V1"]
# print(f"number of verse names: {len(unique_bkvs)}")

# # Initialize tqdm for the progress bar
# total_bkvs = len(unique_bkvs)

# for bkv in unique_bkvs:
#       process_bkv(
#             bkv,
#             DATA_PATH + "occurrences",
#             verses_df,
#             words_df,
#             blacklist_dict,
#             overwrite,
#         )

In [ ]:
overwrite = True

# Get unique values from the 'bkv' column
unique_bkvs = verses_df["bkv"].unique()
print(f"number of verse names: {len(unique_bkvs)}")

# Initialize tqdm for the progress bar
total_bkvs = len(unique_bkvs)

In [ ]:
# Execute tasks and gather results
with concurrent.futures.ProcessPoolExecutor() as executor:
    # Submit tasks and collect futures
    futures = [
        executor.submit(
            process_bkv,
            bkv,
            DATA_PATH + "occurrences",
            verses_df,
            words_df,
            blacklist_dict,
            overwrite,
        )
        for bkv in unique_bkvs
    ]

    progress_bar = tqdm(total=total_bkvs, desc="Processing")

    # Gather results
    for future in concurrent.futures.as_completed(futures):
        progress_bar.update(1)  # Update the progress bar

    # Close the progress bar
    progress_bar.close()

## 4 Merging multiple csv files to one
Concatenating with `awk` (obviously it needs to be installed) is done, as we know all files do have the same header. Also, this is computationally more efficient than first reading each file into a pd.DataFrame and then merging those into one.

In [ ]:
import subprocess

# Build the command string
command = f"awk 'FNR==1 && NR!=1 {{ next; }} {{ print }}' {DATA_PATH}occurrences/B*.csv > {TMP_PATH}occurrences.csv"

# Execute the command
try:
    subprocess.run(command, shell=True, check=True)
    print("Command executed successfully!")
except subprocess.CalledProcessError as e:
    print(f"Error occurred while executing the command: {e}")

## 5 Cleanup

In [ ]:
# read data
occurrences_df = pd.read_csv(
    TMP_PATH + "occurrences.csv",
    dtype={
        "verse_id": "Int64",
        "variantID": "Int64",
        "occurrence": "boolean",
        "wordID": "Int64",
    },
)
# drop rows with empty cells for occurrence or wordID
occurrences_df.dropna(subset=["occurrence", "wordID"], inplace=True)
# fill cells with null values
occurrences_df.fillna(value={"variantID": -1}, inplace=True)
# write to file
occurrences_df.to_csv(TMP_PATH + "occurrences.csv", index=False)